In [ ]:
### pip intall ultralytics

In [1]:
from ultralytics import YOLO
import cv2
import json
import datetime

In [2]:
#import the model
model = YOLO("trained.pt")

In [4]:
#list of images path
# this only show the object returned from the model
image_paths = ["after.jpg", "before1.jpg", "afterblak.jpg", "beforeblak.jpg"]
results = model(image_paths)
print(results[0].boxes)


0: 640x640 1 broken_lamp, 316.2ms
1: 640x640 1 dent, 316.2ms
2: 640x640 (no detections), 316.2ms
3: 640x640 1 dent, 316.2ms
Speed: 10.0ms preprocess, 316.2ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([4.])
conf: tensor([0.4035])
data: tensor([[6.1559e+01, 1.2919e+02, 6.5902e+02, 4.0590e+02, 4.0348e-01, 4.0000e+00]])
id: None
is_track: False
orig_shape: (897, 750)
shape: torch.Size([1, 6])
xywh: tensor([[360.2892, 267.5445, 597.4604, 276.7165]])
xywhn: tensor([[0.4804, 0.2983, 0.7966, 0.3085]])
xyxy: tensor([[ 61.5590, 129.1863, 659.0194, 405.9027]])
xyxyn: tensor([[0.0821, 0.1440, 0.8787, 0.4525]])


In [5]:
#ONLY INFO no image change the last variable for custom images
# you can use the save=True in the YOLO() to save the images
def run_yolo(model_path, image_path):
    """Run YOLO on a single image and return damage detections."""
    model = YOLO(model_path)
    # this can be changed to return the full list just remove "[0]"
    results = model(image_path)[0]
    
    detections = []

    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        detections.append({
            "class_id": cls,
            "label": results.names[cls],
            "confidence": round(conf, 3),
            "bbox": [round(x1), round(y1), round(x2), round(y2)]
        })

    return detections


def compare_damages(pickup_dmg, return_dmg, iou_threshold=0.4):
    """Compare YOLO detections between pickup and return images."""

    def iou(boxA, boxB):
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])

        inter = max(0, xB - xA) * max(0, yB - yA)
        if inter == 0:
            return 0

        areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

        return inter / float(areaA + areaB - inter)

    new_damage = []
    missing_damage = []
    unchanged = []

    for ret in return_dmg:
        matched = False
        for pick in pickup_dmg:
            if iou(ret["bbox"], pick["bbox"]) > iou_threshold:
                unchanged.append(ret)
                matched = True
                break
        if not matched:
            new_damage.append(ret)

    for pick in pickup_dmg:
        matched = False
        for ret in return_dmg:
            if iou(ret["bbox"], pick["bbox"]) > iou_threshold:
                matched = True
                break
        if not matched:
            missing_damage.append(pick)

    return {
        "new_damage_detected": new_damage,
        "missing_previous_damage": missing_damage,
        "unchanged_damage": unchanged
    }


if __name__ == "__main__":
    model_path = "trained.pt"
    pickup_img = "afterblak.jpg"
    return_img = "beforeblak.jpg"

    print("Running YOLO on pickup image...")
    pickup_detections = run_yolo(model_path, pickup_img)

    print("Running YOLO on return image...")
    return_detections = run_yolo(model_path, return_img)

    print("Comparing...")
    comparison = compare_damages(pickup_detections, return_detections)

    print(json.dumps(comparison, indent=2))

Running YOLO on pickup image...

image 1/1 C:\Users\Joker\Desktop\aspire rental damage detection\python model\final version\model in a notebook\afterblak.jpg: 384x640 (no detections), 187.0ms
Speed: 1.1ms preprocess, 187.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
Running YOLO on return image...

image 1/1 C:\Users\Joker\Desktop\aspire rental damage detection\python model\final version\model in a notebook\beforeblak.jpg: 416x640 1 dent, 191.3ms
Speed: 1.2ms preprocess, 191.3ms inference, 0.9ms postprocess per image at shape (1, 3, 416, 640)
Comparing...
{
  "new_damage_detected": [
    {
      "class_id": 0,
      "label": "dent",
      "confidence": 0.825,
      "bbox": [
        153,
        138,
        219,
        191
      ]
    }
  ],
  "missing_previous_damage": [],
  "unchanged_damage": []
}


In [6]:
#saves images with custom boxes and annotations
# check the variables at the end to use custom images
def run_yolo(model_path, image_path):
    """Run YOLO on an image and return detections and raw image."""
    model = YOLO(model_path)
    results = model(image_path)[0]  # only first image
    img = cv2.imread(image_path)
    h, w, _ = img.shape

    detections = []

    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        detections.append({
            "class_id": cls,
            "label": results.names[cls],
            "confidence": round(conf, 3),
            "bbox": [int(x1), int(y1), int(x2), int(y2)]
        })

    # Load image with OpenCV
    
    return detections, img

def draw_boxes(img, detections, color=(0,255,0), thickness=2):
    """Draw bounding boxes with labels on image."""
    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        label = f"{det['label']} {det['confidence']}"
        cv2.rectangle(img, (x1,y1), (x2,y2), color, thickness)
        cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / float(areaA + areaB - inter)

def compare_and_draw(pickup_det, return_det, pickup_img, return_img, iou_thresh=0.4):
    new_damage = []
    unchanged = []

    # check new damage
    for ret in return_det:
        matched = False
        for pick in pickup_det:
            if iou(ret["bbox"], pick["bbox"]) > iou_thresh:
                unchanged.append(ret)
                matched = True
                break
        if not matched:
            new_damage.append(ret)

    # draw boxes
    pickup_img = draw_boxes(pickup_img, pickup_det, color=(0,255,0))  # green = pickup
    return_img = draw_boxes(return_img, unchanged, color=(0,255,255))  # yellow = unchanged
    return_img = draw_boxes(return_img, new_damage, color=(0,0,255))   # red = new damage

    # save images
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    pickup_filename = f"pickup_annotated_{timestamp}.jpg"
    return_filename = f"return_annotated_{timestamp}.jpg"
    cv2.imwrite(pickup_filename, pickup_img)
    cv2.imwrite(return_filename, return_img)

    return new_damage, unchanged

if __name__ == "__main__":
    #to use custome images add your image path
    model_path = "trained.pt"
    pickup_img_path = "afterblak.jpg"
    return_img_path = "beforeblak.jpg"

    pickup_det, pickup_img = run_yolo(model_path, pickup_img_path)
    return_det, return_img = run_yolo(model_path, return_img_path)

    new_damage, unchanged = compare_and_draw(pickup_det, return_det, pickup_img, return_img)

    print(f"New damage detected: {len(new_damage)}")
    print(f"Unchanged damage: {len(unchanged)}")
    print("Annotated images saved: pickup_annotated.jpg, return_annotated.jpg")


image 1/1 C:\Users\Joker\Desktop\aspire rental damage detection\python model\final version\model in a notebook\afterblak.jpg: 384x640 (no detections), 164.5ms
Speed: 1.7ms preprocess, 164.5ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 C:\Users\Joker\Desktop\aspire rental damage detection\python model\final version\model in a notebook\beforeblak.jpg: 416x640 1 dent, 181.9ms
Speed: 1.2ms preprocess, 181.9ms inference, 0.9ms postprocess per image at shape (1, 3, 416, 640)
New damage detected: 1
Unchanged damage: 0
Annotated images saved: pickup_annotated.jpg, return_annotated.jpg


In [ ]:
#testing fast api back end functionality
import requests

url = "http://127.0.0.1:8000/upload-images"
API_URL = "http://127.0.0.1:8000/compare-batch"
files = [
    # --- PICKUP IMAGES (Form Field: pickup_images) ---
    ('pickup_images', ('after.jpg', open('after.jpg', 'rb'), 'image/jpeg')),
    ('pickup_images', ('afterblak.jpg', open('afterblak.jpg', 'rb'), 'image/jpeg')),
    
    # --- RETURN IMAGES (Form Field: returned_images) ---
    ('returned_images', ('before1.jpg', open('before1.jpg', 'rb'), 'image/jpeg')),
    ('returned_images', ('beforeblak.jpg', open('beforeblak.jpg', 'rb'), 'image/jpeg')),
]

response = requests.post(API_URL, files=files)
print(response)
print(response.json())